# FFHQ-Wrinkle: Exploratory Data Analysis

This notebook checks image–mask pairing, dimensions, label density, and representative samples before segmentation training. It does not modify the dataset.

In [ ]:
from pathlib import Path
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image

DATA_ROOT = Path(r"D:/Aphrodize/storage/data/non_time_serie/ffhq_wrinkle")
OUTPUT_DIR = Path(r"D:/Aphrodize/storage/artifacts/non_time_serie/ffhq_wrinkle_eda")
IMAGES_ROOT = DATA_ROOT / "images1024x1024"
MASKS_ROOT = DATA_ROOT / "manual_wrinkle_masks"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

assert IMAGES_ROOT.is_dir(), f"Missing: {IMAGES_ROOT}"
assert MASKS_ROOT.is_dir(), f"Missing: {MASKS_ROOT}"

In [ ]:
def matching_image_path(mask_path: Path) -> Path:
    image_id = int(mask_path.stem)
    batch = (image_id // 1000) * 1000
    return IMAGES_ROOT / f"{batch:05d}" / mask_path.name

records, missing = [], []
for mask_path in sorted(MASKS_ROOT.glob("*.png")):
    image_path = matching_image_path(mask_path)
    if not image_path.exists():
        missing.append({"image_id": mask_path.stem, "expected_image_path": str(image_path)})
        continue
    with Image.open(mask_path) as mask_file:
        mask = np.asarray(mask_file.convert("L"))
    with Image.open(image_path) as image_file:
        image_size = image_file.size
    records.append({
        "image_id": mask_path.stem,
        "image_path": str(image_path),
        "mask_path": str(mask_path),
        "image_size": image_size,
        "mask_size": (mask.shape[1], mask.shape[0]),
        "size_matches": image_size == (mask.shape[1], mask.shape[0]),
        "unique_mask_values": np.unique(mask).tolist(),
        "wrinkle_pixels": int((mask > 0).sum()),
        "total_pixels": int(mask.size),
        "wrinkle_area_ratio": float((mask > 0).mean()),
    })

metrics = pd.DataFrame(records)
missing_df = pd.DataFrame(missing)
print(f"Manual masks: {len(records) + len(missing)}")
print(f"Matched pairs: {len(metrics)}")
print(f"Missing source images: {len(missing_df)}")
metrics.head()

In [ ]:
summary = {
    "manual_masks_found": len(records) + len(missing),
    "matched_image_mask_pairs": len(metrics),
    "missing_source_images": len(missing_df),
    "all_dimensions_match": bool(metrics["size_matches"].all()),
    "wrinkle_area_ratio": metrics["wrinkle_area_ratio"].describe()[["min", "mean", "50%", "max"]].to_dict(),
}
summary

In [ ]:
plt.figure(figsize=(8, 5))
plt.hist(metrics["wrinkle_area_ratio"], bins=30, color="#8b5cf6", edgecolor="white")
plt.xlabel("Wrinkle area ratio")
plt.ylabel("Number of images")
plt.title("FFHQ-Wrinkle manual-mask distribution")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "wrinkle_area_histogram.png", dpi=150)
plt.show()

In [ ]:
sample = metrics.sample(n=min(12, len(metrics)), random_state=42).reset_index(drop=True)
fig, axes = plt.subplots(len(sample), 3, figsize=(12, 4 * len(sample)))
for index, row in sample.iterrows():
    with Image.open(row.image_path) as image_file:
        image = np.asarray(image_file.convert("RGB"))
    with Image.open(row.mask_path) as mask_file:
        mask = np.asarray(mask_file.convert("L"))
    overlay = image.copy()
    overlay[mask > 0] = [255, 60, 60]
    for axis, display, title in zip(axes[index], [image, mask, overlay], ["Image", "Mask", "Overlay"]):
        axis.imshow(display, cmap="gray" if display.ndim == 2 else None)
        axis.set_title(title)
        axis.axis("off")
    axes[index, 0].set_ylabel(f"{row.image_id}\n{row.wrinkle_area_ratio:.3%}", rotation=0, labelpad=55, va="center")
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "sample_pairs.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
metrics.to_csv(OUTPUT_DIR / "manual_mask_metrics.csv", index=False)
missing_df.to_csv(OUTPUT_DIR / "missing_images.csv", index=False)
(OUTPUT_DIR / "summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
print(f"Saved EDA outputs to: {OUTPUT_DIR}")